In [1]:
import os
from collections.abc import Sequence
import csv
import pickle
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import sinter
from sympy.abc import x, y
from qldpc import circuits, codes, decoders
from qldpc.objects import Pauli, PauliXZ

In [2]:
def get_label(
    code: codes.ClassicalCode | codes.QuditCode,
    distance_trials: bool | int = False,
) -> str:
    """Get a label for a code in a figure.

    If the code distance is not known, use a given number of trials to estimate it.
    """
    known_distance = code.get_distance_if_known()
    if isinstance(known_distance, int):
        return f"$d={known_distance}$"
    if not distance_trials:
        return f"[{len(code)}, {code.dimension}]"
    distance_estimate = code.get_distance_bound(num_trials=int(distance_trials))
    return f"[{len(code)}, {code.dimension}, d <= {distance_estimate}]"

In [3]:
def run_memory_experiments(
    codes_to_simulate: Sequence[codes.ClassicalCode | codes.CSSCode],
    basis: PauliXZ,
    error_rates: Sequence[float] = list(np.logspace(-4, -2, 5)),
    max_shots: int = 10**6,
    max_errors: int = 100,
    distance_trials: int = 100,
    **decoding_kwargs: object,
) -> list[sinter.TaskStats]:
    """Use sinter to simulate memory cycles of the given codes.

    This function ...
      1. Builds a memory experiment for each code.
      2. Adds a depolarizing noise to all gates, at the provided error rates.
      2. Wraps each noisy circuit in a `sinter.Task``.
      3. Runs batched Monte Carlo sampling with `sinter.collect`.

    Args:
        codes_to_simulate: The codes we wish to simulate.
        basis: The type of logical operator whose errors are tracked.
        error_rates: The i.i.d. probabilities of a depolarizing error after each gate.
        max_shots: Stop sampling the circuit after this many shots.
        max_errors: Stops the sampling process after this many errors have been seen in samples
            taken from the circuit.
        distance_trials: If the code distance is not known, estimate it with this many trials.
            The estimated distance determines the number of rounds of syndrome measurement.
        **decoding_kwargs: Arguments to pass to the decoder.

    Returns:
        A `sinter` stats object aggregating shots, errors, and metadata for downstream plotting.
    """
    noise_models = {
        prob: circuits.DepolarizingNoiseModel(prob, include_idling_error=False)
        for prob in error_rates
    }

    tasks = []
    for code in codes_to_simulate:
        distance = code.get_distance(bound=distance_trials)
        circuit = circuits.get_memory_experiment(code, basis=basis, num_rounds=distance)
        for prob in error_rates:
            noisy_circuit = noise_models[prob].noisy_circuit(circuit)
            tasks.append(
                sinter.Task(
                    circuit=noisy_circuit,
                    json_metadata={"label": get_label(code), "prob": prob},
                )
            )

    decoder = decoders.SinterDecoder(**decoding_kwargs)
    return sinter.collect(
        num_workers=os.cpu_count() - 2,
        max_shots=max_shots,
        max_errors=max_errors,
        tasks=tasks,
        decoders=["custom"],
        custom_decoders={"custom": decoder},
    )


def make_memory_experiment_figures(
    codes_to_simulate: Sequence[codes.ClassicalCode | codes.CSSCode],
    error_rates: Sequence[float] = list(np.logspace(-4, -2, 5)),
    max_shots: int = 10**6,
    max_errors: int = 100,
    distance_trials: int = 100,
    figsize: tuple[int, int] = (8, 4),
    **decoding_kwargs: object,
) -> tuple[plt.Figure, plt.Axes]:
    """Use sinter to simulate memory cycles of the given codes, and plot the results.

    Args:
        codes_to_simulate: The codes we wish to simulate.
        error_rates: The i.i.d. probabilities of a depolarizing error after each gate.
        max_shots: Stop sampling the circuit after this many shots.
        max_errors: Stops the sampling process after this many errors have been seen in samples
            taken from the circuit.
        distance_trials: If the code distance is not known, estimate it with this many trials.
            The estimated distance determines the number of rounds of syndrome measurement.
        figsize: The size of the figure to produce.
        **decoding_kwargs: Arguments to pass to the decoder.

    Returns:
        The matplotlib.pyplot figure and axis.
    """
    
    # collect statistics for Z and X logicals (which are corrupted, respectively, by X and Z errors)
    stats_z = run_memory_experiments(
        codes_to_simulate,
        Pauli.Z,
        error_rates,
        max_shots,
        max_errors,
        distance_trials,
        **decoding_kwargs,
    )
    stats_x = run_memory_experiments(
        codes_to_simulate,
        Pauli.X,
        error_rates,
        max_shots,
        max_errors,
        distance_trials,
        **decoding_kwargs,
    )

    # plot the statistics collected above
    figure, axes = plt.subplots(1, 2, sharey=True, figsize=figsize)
    for axis, stats, basis in zip(axes, [stats_z, stats_x], ["Z", "X"]):
        sinter.plot_error_rate(
            ax=axis,
            stats=stats,
            x_func=lambda stats: stats.json_metadata["prob"],
            group_func=lambda stats: stats.json_metadata["label"],
        )
        axis.axline(
            (0, 0),
            slope=1,
            color="k",
            linestyle=":",
            label=r"$p_{\mathrm{log}}=p_{\mathrm{phys}}$",
        )
        axis.set_title(f"{basis} logicals")
        axis.set_xlabel("physical error rate")

    for axis in axes:
        axis.loglog()
        axis.grid(which="both")
    axes[1].legend()
    axes[0].set_ylabel("logical error rate")
    figure.tight_layout()

    return figure, axes, stats_x, stats_z

In [4]:
def process_stats(stats_list, save_csv=True, filename="output.csv"):
    grouped = defaultdict(list)

    # Group by label
    for stat in stats_list:
        label = stat.json_metadata["label"]
        grouped[label].append(stat)

    result = {}

    # Compute x (prob) and y (logical error rate)
    for label, stats in grouped.items():
        x_list = []
        y_list = []

        for stat in stats:
            prob = float(stat.json_metadata["prob"])
            shots = stat.shots
            errors = getattr(stat, "errors", 0)
            discards = getattr(stat, "discards", 0)

            effective_shots = shots - discards if shots - discards > 0 else 1
            logical_error_rate = errors / effective_shots

            x_list.append(prob)
            y_list.append(logical_error_rate)

        # Sort by x for consistency
        paired = sorted(zip(x_list, y_list))
        x_list, y_list = zip(*paired)

        result[label] = (list(x_list), list(y_list))

    # Save to CSV
    if save_csv:
        with open(filename, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["label", "prob", "logical_error_rate"])

            for label, (x_list, y_list) in result.items():
                for x, y in zip(x_list, y_list):
                    writer.writerow([label, x, y])

    return result

### Run Simulation 

In [5]:

with open("deform_code/BB_72_GeneCS.pkl", "rb") as f:
    data = pickle.load(f)

qcode = data["qcode"]

hz = qcode.hz.toarray()
code_1 = codes.ClassicalCode(hz, field=2)   
hx = qcode.hx.toarray()
code_2 = codes.ClassicalCode(hx, field=2)   
deform_code_1 = codes.CSSCode(code_1, code_2)

with open(f"deform_code/BB_72_Gauge.pkl", "rb") as f:
    data = pickle.load(f)

qcode = data["qcode"]

hz = qcode.hz.toarray()
code_1 = codes.ClassicalCode(hz, field=2)   
hx = qcode.hx.toarray()
code_2 = codes.ClassicalCode(hx, field=2)   
deform_code_2 = codes.CSSCode(code_1, code_2)

/home/junyuzh/LDPC/qldpc/.venv/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [6]:
codes_to_simulate = [deform_code_1, deform_code_2]

decoding_kwargs = dict(
    with_BP_LSD=True, max_iter=30, bp_method="ms", lsd_method="lsd_cs", lsd_order=0
)
fig, axis, stats_x, stats_z = make_memory_experiment_figures(codes_to_simulate, max_shots=10**6, **decoding_kwargs)
fig.savefig(f"fig/GeneCS_Sim.png", dpi=300)